Для начала исследования набора данных необходимо импортировать и прочитать датасет. Будем использовать датасет `gemstock_clean`, так как `kolvika_clean` гораздо менее информативна.

Посмотрим на размеры набора данных и типы данных, которые есть в нём на данный момент.

In [2]:
import pandas as pd

df2 = pd.read_csv('C://data_parser/data/clean_data/gemstock_clean.csv')

print(df2.shape)
print(df2.head().dtypes)
print(df2.head())

(5930, 16)
gem_weight          object
gem_completeness    object
commercial_color    object
optical_effect      object
color_fastness      object
gem_sertificate     object
gem_clarity         object
gem_origin          object
gem_enhancement     object
gem_cutting         object
gem_size            object
gem_cut_quality     object
gem_main_color      object
gem_price           object
Ссылка на камень    object
gem_name            object
dtype: object
  gem_weight  gem_completeness             commercial_color  \
0    4.54 ct  одиночный камень          Golden,Vivid Yellow   
1    5.05 ct  одиночный камень  Vivid Red,Pigeons Blood Red   
2   15.04 ct  одиночный камень          Golden,Vivid Yellow   
3    1.21 ct  одиночный камень            Vivid orangy-Pink   
4    5.26 ct  одиночный камень                    Neon Blue   

              optical_effect color_fastness     gem_sertificate  \
0  Флюоресценция (умеренная)  Цвет стабилен  Bellerophon Gemlab   
1    Флюоресценция (сильная)  

Как видно из вывода, мой датасет содержит 5930 записей и 16 столбцов.

Первой проблемой, которая бросается в глаза, является то, что все данные приравнены к `object`.

Второй проблемой, является то, что в серии `gem_size` размеры перечислены. Их лучше разбить.

Третьей проблемой является вид `gem_price`. В данный момент там содержится информация об общей цене на камень и одновременно цене за карат, лучше будет разбить данную серию на отдельные, допустим `gem_price`, `gem_price_per_ct` и привести к числовому типу данных.

Четвёртой проблемой является вид `gem_name` в котором ожидается только название камня, но прописаны ещё и его характеристики, это плохо, нужно будет оставить только название.

Пятой проблемой является вид `gem_enhancement` в котором необходимо привести названия обработки к общему виду на латинице, а также скорее всего разбить на отдельные серии, ведь там могут встречаться сразу несколько обработок на один камень.

Теперь рассмотрим по отдельности каждый столбец, чтобы понять, какие значения и их количество он содержит.

In [3]:
print(df2['gem_enhancement'].value_counts().head(50))

gem_enhancement
отсутствует                                                                          4446
низкотемпературная обработка                                                          912
заполнение маслом (незначительная степень)                                            262
термообработка                                                                        165
заполнение маслом (степень умеренная)                                                  49
заполнение маслом (поверхностное)                                                      11
отсутствует, термообработка                                                             2
заполнение маслом (поверхностное), заполнение маслом (незначительная степень)           2
отсутствует, низкотемпературная обработка                                               1
заполнение полимером (степень значительная)                                             1
заполнение маслом (от незначительной до умеренной степени)                          

Здесь мы видим как раз значение в серии `gem_enhancement` и основную проблему, их может быть несколько, они могут отсутствовать(4446) и их необходимо привести к чему-то общему. \
Думаю, что можно разделить `gem_enhancement` на несколько серий:
- во первых `is_treated`, будет содержать бинарное значение, зависящее от того, облагораживался ли камень;
- во вторых `treatmend_method`, будет содержать наименование метода облагораживания;
- в третьих `treatmend_degree`, будет содержать степень облагораживания, так как в значениях во многих местах указаны степени и они явно будут влиять на цену камня.

In [4]:
print(df2['gem_name'].value_counts().head(50))

gem_name
Лот из граната Родолит в огранке круг 4,26 карата, Танзания                  3
Сет аквамаринов в огранке кушон 1,48 карат, Мадагаскар                       3
Сапфир Royal Blue в огранке овал 0,82 карат, Шри-Ланка                       2
Гранат цаворит Intense Green в огранке кушон 1,69 карата, Танзания           2
Сет аквамаринов в огранке октагон 2,87 карат, Мадагаскар                     2
Радужный лунный камень в огранке сердце 7,04 карат, Мадагаскар               2
Негретый розовый сапфир в огранке овал 2,05 карат, Шри-Ланка                 2
Негретый яркий розово-красный рубин в огранке кушон 2,55 карат, Мозамбик     2
Сапфир Royal Blue в огранке кушон 3,11 карат, Шри-Ланка                      2
Негретый рубин в огранке овал 2,01 карат, Мозамбик                           2
Сапфир Royal Blue в огранке кушон 5,01 карат, Шри-Ланка                      2
Пурпурный гранат в огранке кушон 4,95 карат, Африка                          2
Негретый сапфир Teal в огранке кушон 7,05 к

Из вывода видно, что серию `gem_name` также необходимо обработать, ведь она содержит не только название камня, а ещё и его характеристики, которые уже вынесены в отдельные серии.

Необходимо будет с помощью регулярных выражений обработать данную серию и оставить только названия камней. 
Думаю, проще всего это будет сделать с помощью списка названий камней, допустим `gem_names = ['изумруд', 'рубин', 'турмалин']` и т.д.

In [5]:
print(df2['gem_completeness'].value_counts().head(50))

gem_completeness
одиночный камень                                   4490
парные камни                                        830
комплект из трех или более                          428
калиброванные камни                                  50
парные камни, калиброванные камни                    17
комплект из трех или более, калиброванные камни      13
одиночный камень, калиброванные камни                 8
парные боковые камни                                  1
Name: count, dtype: int64


Здесь всплывает ещё одна проблема.. Как видно из вывода выше, комплектность - `gem_completeness`, также может содержать сразу несколько значений и это нужно будет предусмотреть. 
Также привести всё к латинице.

In [6]:
print(df2['gem_price'].value_counts().head(50))

gem_price
44 100 ₽     17
8 900 ₽      16
17 700 ₽     15
13 300 ₽     15
96 900 ₽     13
52 900 ₽     12
35 300 ₽     12
26 500 ₽     11
66 100 ₽     11
39 700 ₽     11
22 100 ₽     11
132 100 ₽    11
79 300 ₽     10
105 700 ₽    10
37 000 ₽     10
7 100 ₽      10
158 500 ₽     9
101 300 ₽     9
57 300 ₽      9
30 900 ₽      8
4 500 ₽       8
228 900 ₽     8
14 100 ₽      8
5 300 ₽       8
6 200 ₽       8
74 900 ₽      7
88 100 ₽      7
10 600 ₽      7
120 600 ₽     7
87 200 ₽      7
162 900 ₽     7
8 000 ₽       7
334 500 ₽     7
48 500 ₽      7
43 200 ₽      7
24 700 ₽      7
41 400 ₽      7
34 400 ₽      6
290 500 ₽     6
33 500 ₽      6
125 900 ₽     6
23 800 ₽      6
277 300 ₽     6
11 500 ₽      6
140 900 ₽     6
281 700 ₽     6
40 500 ₽      6
30 000 ₽      6
20 300 ₽      6
49 300 ₽      6
Name: count, dtype: int64


В выводе информации по серии `gem_price` видно, что во первых, цена ещё не приведена к числовому типу данных, во вторых, встречаются одинаковые цены.. 

В третьих, есть камни, где отдельно рассчитана цена за карат, а есть те, где только общая цена, следовательно цену за карат будет необходимо дополнительно рассчитывать при разделении серии надвое.

Необходимо будет разделить серию надвое, привести к числовому типу данных обе серии, т.е. общую цену в числовое и цену за карат, рассчитываемую на основе общей и веса камня, также в числовое. 



In [7]:
print(df2['gem_clarity'].value_counts().head(50))

gem_clarity
визуально чистый                                                                      3955
с едва заметными включениями                                                           870
с заметными включениями                                                                595
визуально чистый и с едва заметными включениями (для пар)                              179
с едва заметными включениями и визуально чистый (для пар)                              110
рутиловый шелк, эффект бархатистости                                                    69
полупрозрачный                                                                          36
непрозрачный                                                                            17
с сильно заметными включениями                                                          12
визуально чистый, с заметными включениями                                               11
визуально чистый, с едва заметными включениями                                

По `gem_clarity` можно сказать следующее:

- Во первых, необходимо будет привести всё к общим значениям на латинице;
- Во вторых, видно, что присутствуют сразу несколько значений и это будет необходимо регулировать;
- В третьих, видно, что присутствуют отдельные отметки в скобках - для пар, т.е. эти параметры зависят от комплектности (`gem_completeness`).



In [8]:
print(df2['gem_cutting'].value_counts().head(50))

gem_cutting
кушон                                2019
овал                                  991
изумрудная (октагон)                  990
груша                                 647
радиант                               378
сердце                                230
круг                                  209
ашер                                   85
кабошон                                76
фантазийная                            45
триллион                               44
маркиз                                 30
багет                                  23
изумрудная (октагон),радиант           21
принцесса                              20
кабошон,овал                           14
ашер,изумрудная (октагон)              12
кабошон,сахарная голова                10
гексагон                               10
сахарная голова                         7
трапеция                                6
кабошон,фантазийная                     6
кайт,фантазийная                        5
кайт                  

По `gem_cutting` сразу бросается в глаза то, что огранок может быть несколько, судя по всему это также зависит от комплектности.

- Во первых, необходимо будет привести всё к общим наименованиям на латинице;
- Во вторых, понять, как разделять сразу несколько огранок в одной записи.


In [9]:
print(df2['gem_cut_quality'].value_counts().head(50))

gem_cut_quality
Очень хорошее (точная огранка)                                                  2167
Превосходное (высокоточная огранка)                                             1687
Превосходное (высокоточная огранка от GS)                                       1076
Хорошее ("азиатская" или "нативная" огранка)                                     844
Удовлетворительное                                                                 2
Very good                                                                          1
Очень хорошее (точная огранка), Хорошее ("азиатская" или "нативная" огранка)       1
Превосходное (высокоточная огранка), Очень хорошее (точная огранка)                1
Name: count, dtype: int64


По `gem_cut_quality` бросается в глаза опять же, что качеств огранки может быть несколько, хоть их и мало(единичные случаи).

- Во первых, привести всё к общим наименованиям на латинице;
- Во вторых, разобраться с редкими случаями нескольких огранок.



In [10]:
print(df2['gem_origin'].value_counts().head(50))

gem_origin
Шри-Ланка (Цейлон)                        1181
Мадагаскар                                1122
Танзания                                   921
Африка                                     535
Мозамбик                                   446
Бирма (Мьянма)                             357
Афганистан                                 270
Колумбия                                   259
Бразилия                                   134
Нигерия                                    129
Таджикистан (Памир)                         64
Вьетнам                                     51
Замбия                                      37
Камбоджа                                    33
Россия                                      30
Урал, Россия                                30
Конго                                       24
Пакистан                                    17
Кения                                       16
Эфиопия                                     14
Малави                                      12
Ав

По `gem_origin` в глаза бросается опять же по несколько происхождений в одной записи.. 

- Во первых, необходимо будет привести всё к общим наименованиям на латинице;
- Во вторых, понять, что лучше оставлять, конкретное месторождение или только страну, либо разделить на отдельные серии страну и конкретное месторождение.

In [11]:
print(df2['commercial_color'].value_counts().head(50))

commercial_color
Royal Blue / Vivid Blue           174
Vivid Green                       154
Vivid Pink                        126
Cornflower Blue / Intense Blue    109
Vivid Yellow                       99
Teal                               93
Neon Blue                          85
Pigeons Blood Red                  69
Neon Greenish Blue                 68
Intense Green                      64
Muzo Green                         64
Baby Pink                          60
Bi-color                           55
Velvety purple color               51
Pastel Blue                        49
Pastel Yellow                      49
Neon Green                         47
Intense Yellow                     45
Fanta                              45
Intense Blue                       39
Vibrant pink-red                   38
Teal,Bi-color                      36
Lavender                           35
Lagoon                             34
Golden,Vivid Yellow                32
Mint                             

По `commercial_color` пока наиболее хороший результат..

Необходимо будет разобраться со случаями, когда сразу несколько цветов встречаются в одной записи.

In [12]:
print(df2['gem_main_color'].value_counts().head(50))

gem_main_color
зеленый                         789
голубой                         730
синий                           534
розовый                         529
желтый                          394
зеленый,голубой                 270
пурпурный,розовый               241
красный                         232
фиолетовый                      169
пурпурный                       165
оранжевый,розовый               161
желтый,зеленый                  160
красный,розовый                 157
зеленый,синий                   139
голубой,синий                   126
оранжевый                       124
синий,фиолетовый                 94
белый (бесцветный)               89
фиолетовый,розовый               49
оранжевый,желтый                 48
зеленый,розовый                  46
красный,пурпурный                42
красный,оранжевый                38
серый                            38
фиолетовый,пурпурный             36
фиолетовый,серый                 36
синий,серый                      21
голубой,серый

По `gem_main_color` бросается в глаза наличие сразу нескольких цветов в одной записи.

- Во первых, привести все цвета к общим наименованиям на латинице;
- Во вторых, понять, как корректно разделять сразу несколько цветов в одной записи.

In [13]:
print(df2['gem_sertificate'].value_counts().head(50))

gem_sertificate
GFCO GEM LAB - Швейцария                                        1023
AGA                                                              302
Bellerophon Gemlab                                               244
ICA GemLab                                                       196
GRS                                                              155
GAIA (powered by ICA GemLab)                                     125
GIA                                                               73
GUILD                                                             34
МГЛ                                                               32
AIGS                                                              28
Lotus Gemology                                                    17
ICA GemLab & GS Index                                             15
GIT (Thailand)                                                    12
Bellerophon Gemlab, GFCO GEM LAB - Швейцария                       7
ICA GemLab & GS In

По `gem_sertificate` можно заметить, что опять же, присутствует сразу по несколько сертификатов на одну запись, нужно будет разобраться и привести все записи к общему виду на латинице.


In [14]:
print(df2['gem_size'].value_counts().head(50))

gem_size
5 × 3 mm                                      29
3 mm                                           7
2.81 × 2.88 × 2.78 mm                          3
8.64 × 7.59 × 6.08 mm                          3
4 × 3 mm                                       3
5×3                                            3
4.26 × 4.27 × 4.25 mm                          3
6 × 4 mm                                       3
6.92 × 6.74 × 5.03 mm                          2
11.55 × 9.10 × 5.93 mm                         2
6.64 × 6.11 × 4.22 mm                          2
8.94 × 6.83 × 5.86 mm                          2
7.99 × 6.71 × 4.15 mm                          2
6.88 × 6.18 × 4.36 mm                          2
8.51 × 7.55 × 5.40 mm                          2
10.99 × 10.16 × 6.02 mm                        2
5.01 × 4.10 × 2.86 мм5.11 × 4.12 × 2.92 мм     2
9.62 × 8.56 × 6.19 mm                          2
9.94 × 7.81 × 4.59 mm                          2
7.69 × 7.31 × 4.70 mm                          2
12.40 × 9.1

По `gem_size`, как я и говорил ранее, необходимо будет разделить размеры на 3 отдельных серии.

Также встречаются размеры только с одним значением, с двумя и соответственно с тремя.

Встречаются ещё и по несколько размеров в одной записи, необходимо будет разделять.

Также приписка `mm` может встречаться на русском, необходимо будет привести всё к типу данных `float64` при разбиении на серии.

In [15]:
print(df2['optical_effect'].value_counts().head(50))

optical_effect
Флюоресценция (сильная)                                       312
Без флюоресценции                                             179
Флюоресценция (умеренная)                                     178
Флюоресценция (слабая)                                         85
Адуляресценция                                                 35
Флюоресценция (очень сильная)                                  32
Цветовой сдвиг                                                 29
Опалесценция                                                   24
Александритовый эффект (смена цвета)                           16
Астеризм (эффект звезды)                                        6
"Кошачий глаз"                                                  5
Плеохроизм                                                      3
"Mariposa" - Butterfly-wing effect                              2
Флюоресценция (умеренная), Цветовой сдвиг                       2
Флюоресценция (сильная), Астеризм (эффект звезды)            

По `optical_effect` опять же бросается в глаза по несколько значений в одной записи.

- Во первых, привести всё к единому виду наименований на латинице;
- Во вторых, разделить `optical_effect` на две серии - непосредственно оптический эффект и его силу;
- В третьих, разобраться, как разделять эффекты в редких случаях, когда их несколько в записи.


In [16]:
print(df2['color_fastness'].value_counts().head(50))

color_fastness
Цвет стабилен                            257
Не подвергался тестированию               28
"Xамелеон" (обратимый цветовой сдвиг)      7
Name: count, dtype: int64


По `color_fastness` в целом ничего такого в глаза не бросается, необходимо будет привести все наименования к общему виду на латинице и обработать пустые значения (`Не подвергался тестированию`)

In [17]:
print(df2['Ссылка на камень'].value_counts().head(50))

Ссылка на камень
https://gemstock.ru/product/10761/                                                                             1
https://gemstock.ru/product/turmalin-oranzhevato-rozovyy-v-ogranke-oktagon-3-22-karat-afrika/                  1
https://gemstock.ru/product/unheated-light-blue-radiant-cut-sapphire-2-53-ct-sri-lanka/                        1
https://gemstock.ru/product/Bright-greenish-blue-tourmaline-Paraiba-from-Mozambique-3-17-ct/                   1
https://gemstock.ru/product/polikhromnyy-indigolit-v-ogranke-oktagon-4-29-karata/                              1
https://gemstock.ru/product/negretyy-rozovo-fioletovyy-sapfir-v-ogranke-kushon-0-96-karat-madagaskar/          1
https://gemstock.ru/product/purpurnyy-granat-iz-tanzanii-v-ogranke-kushon-4-90-karat/                          1
https://gemstock.ru/product/Unheated-cornflower-blue-sapphire-cushion-cut-1-74-ct-Sri-Lanka/                   1
https://gemstock.ru/product/sero-fioletovaya-birmanskaya-shpinel-v-ogranke-kush

По `Ссылка на камень` необходимо будет удалить, т.к. она не представляет никакой ценности после извлечения данных со странички.

Общие выводы:

- Наиболее часто встречается проблема наличия сразу нескольких значений в одной записи, необходимо будет разобраться, как её решить для каждой серии корректно;
- Все значения всё ещё не приведены к конкретным типам данных и хранятся как `object`, что также необходимо будет исправить и привести серии к нужным типам данных;
- Почти все значения на русском языке, необходимо привести к более лаконичным и кратким аналогиям на латинице.
- Необходимо будет обработать пропуски и удалить ненужную серию `Ссылка на камень`.

Начну с того, что приведу все значения в датафрейме к определённым типам данных.

In [18]:
print(f'До преобразования во float:\n{df2['gem_weight'].head()}')

print(f'\nПосле преобразования во float:\n{df2['gem_weight'].str.replace("ct", "", regex=False).str.replace(" ", "", regex=False).astype(float)}')

До преобразования во float:
0     4.54 ct
1     5.05 ct
2    15.04 ct
3     1.21 ct
4     5.26 ct
Name: gem_weight, dtype: object

После преобразования во float:
0        4.54
1        5.05
2       15.04
3        1.21
4        5.26
        ...  
5925     1.16
5926     1.24
5927     2.07
5928     2.63
5929     2.50
Name: gem_weight, Length: 5930, dtype: float64


Но в данном примере `pandas` создаёт копию исходной серии и не модифицирует `gem_weight` в `df2`. 

Теперь, после примера, необходимо зафиксировать преобразование в наборе данных.


In [19]:
df2['gem_weight'] = (
    df2['gem_weight']
    .str.replace('ct', '', regex=False)
    .str.replace(' ', '', regex=False)
    .astype(float)
)

Проверим, что изменения вступили в силу:

In [20]:
print(df2['gem_weight'].head())

0     4.54
1     5.05
2    15.04
3     1.21
4     5.26
Name: gem_weight, dtype: float64


Данное преобразование открывает возможности для других вычислений и операций.

В следующем примере вычисляются средний, максимальный и минимальный веса камней:

In [19]:
print('Средний вес: ', df2['gem_weight'].mean())
print('Максимальный вес: ', df2['gem_weight'].max())
print('Минимальный вес: ', df2['gem_weight'].min())

Средний вес:  5.123566260987153
Максимальный вес:  202.0
Минимальный вес:  0.22


Далее преобразуем серию `gem_price`:

In [21]:
print(df2['gem_price'].head(10), end='\n\n')

print(df2['gem_price']
      .str.split(pat='/', n=1)
      .str.get(0)
      .str.replace('₽', '', regex=False)
      .str.replace(' ', '', regex=False)
      .astype(float))


0       1 982 700 ₽/ 436,718/ct
1    17 230 500 ₽/ 3,411,980/ct
2       4 711 800 ₽/ 313,285/ct
3         176 600 ₽/ 145,950/ct
4    25 458 500 ₽/ 4,840,019/ct
5       2 319 500 ₽/ 418,682/ct
6       3 147 200 ₽/ 343,206/ct
7          220 200 ₽/ 79,209/ct
8       4 077 200 ₽/ 399,725/ct
9       1 464 600 ₽/ 362,525/ct
Name: gem_price, dtype: object

0        1982700.0
1       17230500.0
2        4711800.0
3         176600.0
4       25458500.0
           ...    
5925        9300.0
5926        5500.0
5927      110100.0
5928       27800.0
5929       33100.0
Name: gem_price, Length: 5930, dtype: float64


После успешного преобразования внесу изменения в исходный набор данных:

In [22]:
df2['gem_price'] = (
    df2['gem_price']
    .str.split(pat='/', n=1)
    .str.get(0)
    .str.replace('₽', '', regex=False)
    .str.replace(' ', '', regex=False)
    .astype(float))

Проверим, что изменения вступили в силу:

In [23]:
print(df2['gem_price'].head(10))

0     1982700.0
1    17230500.0
2     4711800.0
3      176600.0
4    25458500.0
5     2319500.0
6     3147200.0
7      220200.0
8     4077200.0
9     1464600.0
Name: gem_price, dtype: float64


После преобразования серий `gem_price` и `gem_weight` преобразуем серию `gem_size`:

In [24]:
import pandas as pd
import numpy as np
import re

def parse_gem_size(size_str):
    # Если пусто, возвращаем NaN
    if pd.isna(size_str):
        return pd.Series([np.nan, np.nan, np.nan, np.nan])
    
    # 1. Нормализация: приводим всё к нижнему регистру, меняем 'м' на 'm', убираем запятые в числах
    s = str(size_str).lower().replace('м', 'm').replace(',', '.')
    
    # 2. Разделяем строку по 'mm', чтобы отделить размеры разных камней
    # Пример: "5.01 × 4.10 × 2.86 mm5.11 × 4.12 × 2.92 mm" -> ['5.01 × 4.10 × 2.86 ', '5.11 × 4.12 × 2.92 ', '']
    stones = s.split('mm')
    
    # Считаем, сколько реально камней описано (убираем пустые куски)
    stones_count = len([x for x in stones if x.strip()])
    if stones_count == 0: 
        stones_count = 1
        
    # Берем размеры ТОЛЬКО ПЕРВОГО камня
    first_stone = stones[0]
    
    # 3. Магия: находим ВСЕ числа в строке первого камня (игнорируя x, ×, пробелы)
    # \d+\.?\d* находит и целые (5), и дробные (9.73) числа
    nums = re.findall(r'\d+\.?\d*', first_stone)
    
    # 4. Распределяем числа в зависимости от их количества
    if len(nums) == 3:
        # Полный формат (Д x Ш x Г)
        return pd.Series([float(nums[0]), float(nums[1]), float(nums[2]), stones_count])
    elif len(nums) == 2:
        # Калибровка (Д x Ш), глубины нет
        return pd.Series([float(nums[0]), float(nums[1]), np.nan, stones_count])
    elif len(nums) == 1:
        # Круглый камень (только диаметр)
        val = float(nums[0])
        return pd.Series([val, val, np.nan, stones_count])
    else:
        return pd.Series([np.nan, np.nan, np.nan, stones_count])

# Применяем функцию ко всей колонке 
size_df = df2['gem_size'].apply(parse_gem_size)
size_df.columns = ['length_mm', 'width_mm', 'depth_mm', 'stones_in_size']

# Присоединяем новые колонки к основному датафрейму
df2 = pd.concat([df2, size_df], axis=1)


In [25]:
print(df2.head(10))

   gem_weight  gem_completeness             commercial_color  \
0        4.54  одиночный камень          Golden,Vivid Yellow   
1        5.05  одиночный камень  Vivid Red,Pigeons Blood Red   
2       15.04  одиночный камень          Golden,Vivid Yellow   
3        1.21  одиночный камень            Vivid orangy-Pink   
4        5.26  одиночный камень                    Neon Blue   
5        5.54  одиночный камень          Golden,Vivid Yellow   
6        9.17  одиночный камень                   Neon Green   
7        2.78  одиночный камень                          NaN   
8       10.20  одиночный камень          Golden,Vivid Yellow   
9        4.04  одиночный камень                          NaN   

              optical_effect color_fastness     gem_sertificate  \
0  Флюоресценция (умеренная)  Цвет стабилен  Bellerophon Gemlab   
1    Флюоресценция (сильная)            NaN                 GRS   
2  Флюоресценция (умеренная)  Цвет стабилен  Bellerophon Gemlab   
3                        Na

После преобразования `gem_size` я получил 4 новых серии:
- `length_mm` - содержащую информацию о длине камня в миллиметрах;
- `width_mm` - содержащую информацию о ширине камня в миллиметрах;
- `depth_mm` - содержащую информацию о глубине(высоте) камня в миллиметрах;
- `stones_in_size` - содержащую информацию по количеству камней, извлечённому из `gem_size`.

Далее удалим серию `Ссылка на камень` за ненадобностью:

In [26]:
del df2['Ссылка на камень']

In [27]:
print(df2.shape)
print(df2.head(10))

(5930, 19)
   gem_weight  gem_completeness             commercial_color  \
0        4.54  одиночный камень          Golden,Vivid Yellow   
1        5.05  одиночный камень  Vivid Red,Pigeons Blood Red   
2       15.04  одиночный камень          Golden,Vivid Yellow   
3        1.21  одиночный камень            Vivid orangy-Pink   
4        5.26  одиночный камень                    Neon Blue   
5        5.54  одиночный камень          Golden,Vivid Yellow   
6        9.17  одиночный камень                   Neon Green   
7        2.78  одиночный камень                          NaN   
8       10.20  одиночный камень          Golden,Vivid Yellow   
9        4.04  одиночный камень                          NaN   

              optical_effect color_fastness     gem_sertificate  \
0  Флюоресценция (умеренная)  Цвет стабилен  Bellerophon Gemlab   
1    Флюоресценция (сильная)            NaN                 GRS   
2  Флюоресценция (умеренная)  Цвет стабилен  Bellerophon Gemlab   
3               

Теперь приступим к нормализации серии `gem_name`, необходимо оставить в ней только названия камней:

In [28]:
print(df2['gem_name'].value_counts())

gem_name
Лот из граната Родолит в огранке круг 4,26 карата, Танзания            3
Сет аквамаринов в огранке кушон 1,48 карат, Мадагаскар                 3
Сапфир Royal Blue в огранке овал 0,82 карат, Шри-Ланка                 2
Гранат цаворит Intense Green в огранке кушон 1,69 карата, Танзания     2
Сет аквамаринов в огранке октагон 2,87 карат, Мадагаскар               2
                                                                      ..
Ярко-розовая шпинель Махенге в огранке кушон 3,52 карата, Танзания     1
Аквамарин в огранке маркиз 0,89 карат, Африка                          1
Ярко-розовая шпинель Махенге в огранке кушон 2,37 карата, Танзания     1
Гранат родолит в огранке кушон 21,56 карата, Танзания                  1
Комплект из морганитов  2,50 карат, Африка                             1
Name: count, Length: 5883, dtype: int64


Для того, чтобы нормализовать серию `gem_name` я составлю список наименований камней, которые в ней присутствуют.

Во время составления этого списка возникло ещё несколько проблем:
- во первых, оказывается в серии помимо камней присутствуют ювелирные изделия, а именно кольца и нужно будет удалить их из серии;
- во вторых, пока ещё не ясно, как обрабатывать пары, сеты и комплекты, поэтому я разделю нормализацию на несколько этапов.

В первом этапе я с помощью функции обработаю все значения с одиночными камнями, а остальные оставлю такими, какие они есть.

Во втором этапе я уже буду думать, как поступить с остальными значениями.

Первый этап: 

In [29]:

raw_gem_names = ['гранат', 'гранат гроссуляр', 'гроссуляр', 'гранат родолит', 'родолит', 'гранат цаворит', 'цаворит', 'гранат спессартин', 'спессартин', 'гранат малайя', 'гранат гессонит', 'гессонит', 'гранат мали', 'морганит', 'шпинель', 'шпинель махенге', 'сапфир', 'сапфир падпараджа', 'полихромный сапфир', 'турмалин', 'турмалин параиба', 'турмалин индиголит', 'индиголит', 'турмалин верделит', 'верделит', 'изумруд', 'изумруд марипоза', 'циркон', 'турмалин рубеллит', 'рубеллит', 'лунный камень', 'радужный лунный камень', 'рубин', 'аквамарин', 'хризоберилл', 'берилл', 'грандидьерит', 'опал', 'опал арлекин', 'аметист', 'цитрин', 'топаз', 'празиолит', 'топаз империал', 'перидот', 'танзанит', 'апатит', 'сфен', 'клиногумит', 'александрит', 'демантоид', 'хризолит', 'кунцит', 'кварц', 'гелиодор']
raw_gem_names.sort(key=len, reverse=True)

print('До нормализации: ')
print(df2['gem_name'].value_counts())

def gem_name_normalize(gem_name_str):
 
    s = str(gem_name_str).lower()

    for raw_name in raw_gem_names:
        if raw_name in s:
            return raw_name
    return s


print('После нормализации: ')
a = df2['gem_name'].apply(gem_name_normalize)
print(a.value_counts())







До нормализации: 
gem_name
Лот из граната Родолит в огранке круг 4,26 карата, Танзания            3
Сет аквамаринов в огранке кушон 1,48 карат, Мадагаскар                 3
Сапфир Royal Blue в огранке овал 0,82 карат, Шри-Ланка                 2
Гранат цаворит Intense Green в огранке кушон 1,69 карата, Танзания     2
Сет аквамаринов в огранке октагон 2,87 карат, Мадагаскар               2
                                                                      ..
Ярко-розовая шпинель Махенге в огранке кушон 3,52 карата, Танзания     1
Аквамарин в огранке маркиз 0,89 карат, Африка                          1
Ярко-розовая шпинель Махенге в огранке кушон 2,37 карата, Танзания     1
Гранат родолит в огранке кушон 21,56 карата, Танзания                  1
Комплект из морганитов  2,50 карат, Африка                             1
Name: count, Length: 5883, dtype: int64
После нормализации: 
gem_name
сапфир                                                             1510
турмалин                    

После первой версии функции для нормализации, удалю значения связанные с ювелирными изделями:

In [45]:
jewelry_keywords = ['кольцо', 'серьги', 'подвеска', 'кулон', 'браслет', 'ring', 'earring', 'pendant', 'брошь', 'кольца', 'броши', 'подвески']

mask_jewelry = df2['gem_name'].str.lower().str.contains('|'.join(jewelry_keywords), na=False)

print(f"Найдено ювелирных изделий: {mask_jewelry.sum()}")
print(df2[mask_jewelry]['gem_name'].head())

df2 = df2[~mask_jewelry].copy()

print(f"Было: {len(df2)}, Стало: {len(df2)}")

Найдено ювелирных изделий: 0
Series([], Name: gem_name, dtype: object)
Было: 5871, Стало: 5871


Уже после выполнения первого этапа я понял одну проблему - списка тех значений, что я использовал для нормализации не хватает, потому что в нём отсутствуют множественные числа для названий камней..

Исходя из этого я решил использовать словарь - структуру данных `dict` вместо списка, ключами которого будут названия камней во множественном и единственном числах, а значениями, доступными по ключу, будут названия камней в единственном числе. 

Благодаря такому словарю количество уникальных значений должно уменьшиться и проблема обработки пар, сетов и комплектов также должна в бОльшей степени решиться.

Перепишу функцию под словарь значений:

In [46]:
gem_mapping = {
    'гранат': 'гранат', 'гранатов': 'гранат', 'гранаты': 'гранат',
    'гроссуляр': 'гранат гроссуляр', 'гроссуляров': 'гранат гроссуляр', 'гроссуляры': 'гранат гроссуляр',
    'родолит': 'гранат родолит', 'родолитов': 'гранат родолит', 'родолиты': 'гранат родолит',
    'цаворит': 'гранат цаворит', 'цаворитов': 'гранат цаворит', 'цавориты': 'гранат цаворит',
    'спессартин': 'гранат спессартин', 'спессартинов': 'гранат спессартин', 'спессартины': 'гранат спессартин',
    'гессонит': 'гранат гессонит', 'гессонитов': 'гранат гессонит', 'гессониты': 'гранат гессонит',
    'морганит': 'морганит', 'морганитов': 'морганит', 'морганиты': 'морганит',
    'шпинель': 'шпинель', 'шпинелей': 'шпинель', 'шпинели': 'шпинель',
    'сапфир': 'сапфир', 'сапфиров': 'сапфир', 'сапфиры': 'сапфир', 'cапфир': 'сапфир', 'sапфир': 'сапфир', 'сафир': 'сапфир',
    'сапфир падпараджа': 'сапфир падпараджа', 'сапфиров падпараджа': 'сапфир падпараджа', 'сапфиры падпараджа': 'сапфир падпараджа',
    'полихромный сапфир': 'полихромный сапфир', 'полихромных сапфиров': 'полихромный сапфир',
    'аквамарин': 'аквамарин', 'аквамаринов': 'аквамарин', 'аквамарины': 'аквамарин',
    'турмалин': 'турмалин', 'турмалинов': 'турмалин', 'турмалины': 'турмалин',
    'турмалин параиба': 'турмалин параиба', 'турмалинов параиба': 'турмалин параиба', 'турмалины параиба': 'турмалин параиба', 
    'параибы': 'турмалин параиба', 'параиба': 'турмалин параиба', 'параибов': 'турмалин параиба',
    'индиголит': 'турмалин индиголит', 'индиголитов': 'турмалин индиголит', 'индиголиты': 'турмалин индиголит',
    'верделит': 'турмалин верделит', 'верделитов': 'турмалин верделит', 'верделиты': 'турмалин верделит',
    'изумруд': 'изумруд', 'изумрудов': 'изумруд', 'изумруды': 'изумруд',
    'изумруд марипоза': 'изумруд марипоза', 'изумрудов марипоза': 'изумруд марипоза', 'изумруды марипоза': 'изумруд марипоза',
    'циркон': 'циркон', 'цирконов': 'циркон', 'цирконы': 'циркон',
    'рубеллит': 'турмалин рубеллит', 'рубеллитов': 'турмалин рубеллит', 'рубеллиты': 'турмалин рубеллит',
    'лунный камень': 'лунный камень', 'лунные камни': 'лунный камень', 'лунных камней': 'лунный камень', 'лунных каменей': 'лунный камень',
    'рубин': 'рубин', 'рубинов': 'рубин', 'рубины': 'рубин',
    'хризоберилл': 'хризоберилл', 'хризобериллов': 'хризоберилл', 'хризобериллы': 'хризоберилл',
    'берилл': 'берилл', 'бериллов': 'берилл', 'бериллы': 'берилл',
    'грандидьерит': 'грандидьерит', 'грандидьеритов': 'грандидьерит', 'грандидьериты': 'грандидьерит',
    'опал': 'опал', 'опалов': 'опал', 'опалы': 'опал', 'опал арлекин': 'опал арлекин',
    'аметист': 'аметист', 'аметистов': 'аметист', 'аметисты': 'аметист',
    'цитрин': 'цитрин', 'цитринов': 'цитрин', 'цитрины': 'цитрин',
    'топаз': 'топаз', 'топазов': 'топаз', 'топазы': 'топаз',
    'топаз империал': 'топаз империал', 'топазов империал': 'топаз империал', 'топазы империал': 'топаз империал',
    'празиолит': 'празиолит', 'празиолитов': 'празиолит', 'празиолиты': 'празиолит',
    'перидот': 'перидот', 'перидотов': 'перидот', 'перидоты': 'перидот',
    'танзанит': 'танзанит', 'танзанитов': 'танзанит', 'танзаниты': 'танзанит',
    'апатит': 'апатит', 'апатитов': 'апатит', 'апатиты': 'апатит',
    'сфен': 'сфен', 'сфенов': 'сфен', 'сфены': 'сфен',
    'клиногумит': 'клиногумит', 'клиногумитов': 'клиногумит', 'клиногумиты': 'клиногумит',
    'александрит': 'александрит', 'александритов': 'александрит', 'александриты': 'александрит',
    'демантоид': 'демантоид', 'демантоидов': 'демантоид', 'демантоиды': 'демантоид',
    'кунцит': 'кунцит', 'кунцитов': 'кунцит', 'кунциты': 'кунцит',
    'умбалит': 'гранат умбалит', 'умбалитов': 'гранат умбалит', 'умбалиты': 'гранат умбалит',
    'хризолит': 'хризолит', 'хризолитов': 'хризолит', 'хризолиты': 'хризолит',
    'пироп': 'пироп', 'пиропов': 'пироп', 'пиропы': 'пироп',
    'кварц': 'кварц', 'кварцев': 'кварц', 'кварцы': 'кварц', 
    'гелиодор': 'гелиодор', 'гелиодоров': 'гелиодор', 'гелиодоры': 'гелиодор',
    'шпинель махенге': 'шпинель махенге', 'шпинелей махенге': 'шпинель махенге', 'шпинели махенге': 'шпинель махенге',
    'хромдиопсид': 'хромдиопсид', 'хромдиопсидов': 'хромдиопсид', 'хромдиопсиды': 'хромдиопсид'
}

sorted_gem_keys = sorted(gem_mapping.keys(), key=len, reverse=True)

def gem_name_normalize(gem_name_str):
   
    s = str(gem_name_str).lower()
    
    for key in sorted_gem_keys:
        if key in s:
            return gem_mapping[key]
            
    return s

a = df2['gem_name'].apply(gem_name_normalize)
print(a.value_counts())


gem_name
сапфир                1499
шпинель                585
турмалин               527
аквамарин              480
изумруд                330
турмалин параиба       312
гранат родолит         250
гранат цаворит         216
шпинель махенге        195
рубин                  194
танзанит               160
циркон                 119
гранат спессартин      112
сапфир падпараджа       86
хризоберилл             75
аметист                 75
лунный камень           70
морганит                60
гранат                  56
турмалин индиголит      52
турмалин рубеллит       52
топаз                   47
опал                    34
сфен                    33
демантоид               30
цитрин                  29
берилл                  25
гранат умбалит          25
перидот                 24
гранат гроссуляр        19
топаз империал          18
турмалин верделит       14
александрит             12
полихромный сапфир      10
празиолит               10
клиногумит               9
кунцит             

Применим изменения к датафрейму:

In [47]:
df2['gem_name'] = a

print(df2['gem_name'].value_counts())

gem_name
сапфир                1499
шпинель                585
турмалин               527
аквамарин              480
изумруд                330
турмалин параиба       312
гранат родолит         250
гранат цаворит         216
шпинель махенге        195
рубин                  194
танзанит               160
циркон                 119
гранат спессартин      112
сапфир падпараджа       86
хризоберилл             75
аметист                 75
лунный камень           70
морганит                60
гранат                  56
турмалин индиголит      52
турмалин рубеллит       52
топаз                   47
опал                    34
сфен                    33
демантоид               30
цитрин                  29
берилл                  25
гранат умбалит          25
перидот                 24
гранат гроссуляр        19
топаз империал          18
турмалин верделит       14
александрит             12
полихромный сапфир      10
празиолит               10
клиногумит               9
кунцит             